In [1]:
import os 
import sys
import random
import numpy as np
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

# SUMO-RL
if "SUMO_HOME" in os.environ:
    tools = os.path.join(os.environ["SUMO_HOME"], "tools")
    sys.path.append(tools)

else:
    sys.exit("Please declare the environment variable 'SUMO_HOME'")

from sumo_rl import SumoEnvironment

print("SUMO-RL imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"SUMO_HOME: {os.environ['SUMO_HOME']}")

SUMO-RL imported successfully!
PyTorch version: 2.10.0+cu128
SUMO_HOME: /usr/share/sumo


In [3]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.net(x)
    
state_dim = 11
action_dim = 4

q_net = QNetwork(state_dim, action_dim)
target_net = QNetwork(state_dim, action_dim)

target_net.load_state_dict(q_net.state_dict())

dummy_state = torch.FloatTensor(np.random.rand(state_dim))
q_values = q_net(dummy_state)

print(f"Q-Network Architecture:\n{q_net}")
print(f"\nInput state shape: {dummy_state.shape}")
print(f"outptut q-values {q_values}")
print(f"Output Q-values shape: {q_values.shape}")
print(f"\nTarget network weights match Q-network: "
      f"{all(torch.equal(p1, p2) for p1, p2 in zip(q_net.parameters(), target_net.parameters()))}")

Q-Network Architecture:
QNetwork(
  (net): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=4, bias=True)
  )
)

Input state shape: torch.Size([11])
outptut q-values tensor([ 0.0363,  0.0915,  0.0351, -0.2101], grad_fn=<ViewBackward0>)
Output Q-values shape: torch.Size([4])

Target network weights match Q-network: True


In [7]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size = 32):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (torch.FloatTensor(np.array(states)),
                torch.LongTensor(np.array(actions)),
                torch.FloatTensor(np.array(rewards)),
                torch.FloatTensor(np.array(next_states)),
                torch.FloatTensor(np.array(dones)))

    def __len__(self):
        return len(self.buffer)
    
buffer = ReplayBuffer(capacity=5000)
for i in range(100):
    state      = np.random.rand(state_dim)
    action     = np.random.randint(action_dim)
    reward     = np.random.uniform(-1, 1)
    next_state = np.random.rand(state_dim)
    done       = False
    buffer.push(state, action, reward, next_state, done)

print(f"Buffer size after 100 pushes: {len(buffer)}")

states, actions, rewards, next_states, dones = buffer.sample(32)

print(f"Buffer Size after 100 pushes: {len(buffer)}")
print(f"\n sampled batch shapes:")
print(f"  States: {states.shape}")
print(f"  Actions: {actions.shape}")
print(f"  Rewards: {rewards.shape}")
print(f"  Next States: {next_states.shape}")
print(f"  Dones: {dones.shape}")


Buffer size after 100 pushes: 100
Buffer Size after 100 pushes: 100

 sampled batch shapes:
  States: torch.Size([32, 11])
  Actions: torch.Size([32])
  Rewards: torch.Size([32])
  Next States: torch.Size([32, 11])
  Dones: torch.Size([32])


In [10]:
class DQNAgent:
    def __init__(self, state_dim, action_dim):
        self.q_net = QNetwork(state_dim, action_dim)
        self.target_net = QNetwork(state_dim, action_dim)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()  # Set target net to evaluation mode

        self.buffer = ReplayBuffer(capacity=5000)
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=0.00025)
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.batch_size = 32

        self.action_dim = action_dim
        self.losses = []

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        else:
            state_tesnor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                q_values = self.q_net(state_tesnor)
            return q_values.argmax().item()
        
    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        current_q = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(1)[0]
            target_q = rewards + self.gamma * max_next_q * (1 - dones)

        loss = nn.MSELoss()(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.losses.append(loss.item())
        return loss.item()
    
    def update_target_network(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

agent = DQNAgent(state_dim = 11, action_dim = 4)

for i in range(100):
    state = np.random.rand(11)
    action = np.random.randint(4)
    reward = np.random.uniform(-1, 1)
    next_state = np.random.rand(11)
    done = False
    agent.buffer.push(state, action, reward, next_state, done)

state = np.random.rand(11)
action = agent.select_action(state)

loss = agent.train_step()

print(f"Agent initialized!!!")
print(f"\nepsilon: {agent.epsilon}")
print(f"buffer size: {len(agent.buffer)}")
print(f"selected action: {action} out off {agent.action_dim} possible actions")
print(f"training loss: {loss:.4f}")
print(f"\nQ-network: {sum(p.numel() for p in agent.q_net.parameters())} parameters")



Agent initialized!!!

epsilon: 1.0
buffer size: 100
selected action: 3 out off 4 possible actions
training loss: 0.3741

Q-network: 5188 parameters


In [ ]:
os.chdir("/home/tapan/classes/CS5180/Final_Project/sumo-rl")
print(f"Working directory: {os.getcwd()}")
net_file = "sumo_rl/nets/2way-single-intersection/single-intersection.net.xml"
print(f"Net file exists: {os.path.exists(net_file)}")



Working directory: /home/tapan/classes/CS5180/Final_Project/sumo-rl
Net file exists: True


In [18]:
import os
os.chdir("/home/tapan/classes/CS5180/Final_Project/sumo-rl")

# Verify
print(f"Working directory: {os.getcwd()}")
print(f"Net file exists: {os.path.exists('sumo_rl/nets/2way-single-intersection/single-intersection.net.xml')}")
print(f"Route file exists: {os.path.exists('sumo_rl/nets/2way-single-intersection/single-intersection-vhvh.rou.xml')}")

seeds = [42, 123, 456, 789, 1000]
total_seconds = 100000
target_update = 10
net_file = "sumo_rl/nets/2way-single-intersection/single-intersection.net.xml"
route_file = "sumo_rl/nets/2way-single-intersection/single-intersection-vhvh.rou.xml"
output_dir = "project/output/vanilla_dqn"

os.makedirs(output_dir, exist_ok=True)

def train_one_seed(seed):
    print(f"\n{'='*40}")
    print(f"Training vanilla DQN with seed: {seed}")
    print(f"\n{'='*40}")
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    env = SumoEnvironment(
        net_file=net_file,
        route_file=route_file,
        out_csv_name=f"{output_dir}/vanilla_dqn_seed_{seed}",
        single_agent=True,
        use_gui=False,
        num_seconds=total_seconds,
    )

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    print(f"state_dim: {state_dim}, action_dim: {action_dim}")

    agent = DQNAgent(state_dim=state_dim, action_dim=action_dim)
    episode = 0
    step = 0

    episode_rewards =[]
    episode_waittimes = []

    state,_ = env.reset()

    while True:
        action = agent.select_action(state)

        # 2. Step environment
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        step += 1

        # 3. Store transition
        agent.buffer.push(state, action, reward, next_state, float(done))

        # 4. Train
        agent.train_step()

        # 5. Move to next state
        state = next_state

        # 6. Episode ended
        if done:
            episode += 1

            # Decay epsilon
            agent.decay_epsilon()

            # Update target network every 10 episodes
            if episode % target_update == 0:
                agent.update_target_network()
                print(f"  Episode {episode} — Target network updated")

            # Log metrics
            avg_wait = info.get("system_total_waiting_time", 0)
            episode_waittimes.append(avg_wait)
            print(f"  Episode {episode:3d} | "
                  f"Steps: {step:6d} | "
                  f"Epsilon: {agent.epsilon:.3f} | "
                  f"Waiting: {avg_wait:.1f}s | "
                  f"Buffer: {len(agent.buffer)}")

            # Reset for next episode
            state, _ = env.reset()

            # Check if training is complete
            if step >= total_seconds:
                break

    # Save model
    torch.save(agent.q_net.state_dict(),
               f"{output_dir}/vanilla_dqn_model_seed_{seed}.pth")
    print(f"\nSeed {seed} complete. Model saved.")

    env.close()
    return episode_waittimes

# ── Run all seeds ────────────────────────
all_results = {}
for seed in seeds:
    all_results[seed] = train_one_seed(seed)

print("\n===== ALL VANILLA DQN TRAINING COMPLETE =====")


Working directory: /home/tapan/classes/CS5180/Final_Project/sumo-rl
Net file exists: True
Route file exists: True

Training vanilla DQN with seed: 42

 Retrying in 1 seconds
Step #0.00 (0ms ?*RT. ?UPS, TraCI: 2ms, vehicles TOT 0 ACT 0 BUF 0)                      
state_dim: 21, action_dim: 4
 Retrying in 1 seconds
Step #99000.00 (6ms ~= 166.67*RT, ~13166.67UPS, TraCI: 18ms, vehicles TOT 54264 ACT 79 BUFtep #4500.00 (0ms ?*RT. ?UPS, TraCI: 12ms, vehicles TOT 2287 ACT 63 BUF 847)             ep #9000.00 (2ms ~= 500.00*RT, ~42000.00UPS, TraCI: 18ms, vehicles TOT 4480 ACT 84 BUF 1p #13500.00 (2ms ~= 500.00*RT, ~38500.00UPS, TraCI: 16ms, vehicles TOT 6573 ACT 77 BUF  #18000.00 (1ms ~= 1000.00*RT, ~62000.00UPS, TraCI: 13ms, vehicles TOT 8858 ACT 62 BUF#22500.00 (2ms ~= 500.00*RT, ~34500.00UPS, TraCI: 14ms, vehicles TOT 11081 ACT 69 BUF27000.00 (3ms ~= 333.33*RT, ~32333.33UPS, TraCI: 22ms, vehicles TOT 13515 ACT 97 BUF1500.00 (3ms ~= 333.33*RT, ~25333.33UPS, TraCI: 17ms, vehicles TOT 16063 AC